# 5. Capstone: Building a FAIR Dataset

This notebook has one goal: take everything from Notebooks 1–4 — FAIR
metadata, EMMO-style concept links, and a NOMAD-style archive/schema split
— and apply the whole pattern, start to finish, to a small dataset you
build yourself. By the end you will have a schema, a validator, a set of
FAIR records, and two exported files (a clean CSV for statistics and a full
JSON archive) — the same shape of deliverable you will produce for real
when you select a dataset for the [course project](../06_project/project.md).

**Scenario**: a set of quenched-and-tempered tool-steel samples, each
heat-treated at a different temperature and time, with measured hardness —
the same kind of process/property data used throughout Parts I–III of this
course, so what you build here plugs directly into later material.

**Topics**
1. Step 1 — Design a schema
2. Step 2 — A FAIR record builder function
3. Step 3 — Generate the dataset
4. Step 4 — Validate every record
5. Step 5 — Export: clean CSV + full JSON archive
6. Step 6 — Round-trip check
7. Step 7 — Dataset-level FAIR check

## Step 1 — Design a Schema

Following Notebook 4's pattern, a schema is just a dict describing which
keys are required and what type each one must be. We separate **process
parameters** (what you controlled) from **results** (what you measured) —
the same `synthesis` / `results` split used in Notebook 4 — and require the
same core FAIR fields as `fair_check` in Notebook 1 at the top level of
every record.

In [ ]:
sample_schema = {
    'identifier':  str,
    'sample_id':   str,
    'creator':     str,
    'created':     str,
    'process': {
        'temperature_C': (int, float),
        'time_h':        (int, float),
        'quench_medium': str,
    },
    'results': {
        'hardness_HV': (int, float),
    },
}

## Step 2 — A FAIR Record Builder Function

Rather than writing each record's dictionary out by hand (error-prone —
Notebook 4's Exercise 2 showed how easy it is to accidentally break a
schema), we write one function that always produces a structurally correct,
EMMO-tagged record. This is the data-handling equivalent of Part I's advice
to package a repeated calculation into a function instead of copy-pasting
it (Part I, Notebook 1, Section 1.4).

One honest wrinkle, worth flagging explicitly: **hardness has no concept in
core EMMO** (you can check this yourself with Notebook 2's
`find_class_by_label` — it returns nothing). EMMO is a *foundational*
ontology; domain-specific mechanical-testing properties like Vickers
hardness belong in a **domain ontology built on top of EMMO**, the same
role BattINFO plays for batteries (Notebook 2, Section 2.4) — at the time
of writing, no widely-adopted public one exists yet for hardness. The
record below uses a clearly-marked local placeholder concept for it instead
of inventing a fake EMMO term, and real EMMO concepts for the two
quantities that *do* exist in EMMO core.

In [ ]:
from datetime import date
import uuid

def make_sample_record(sample_id, temperature_C, time_h, quench_medium, hardness_HV,
                        creator='A. Lindqvist'):
    """Build one FAIR, schema-conformant, EMMO-tagged sample record."""
    return {
        'identifier':  f'urn:uuid:{uuid.uuid4()}',
        'sample_id':   sample_id,
        'creator':     creator,
        'created':     date.today().isoformat(),
        'process': {
            'temperature_C': temperature_C,
            'time_h':        time_h,
            'quench_medium': quench_medium,
            'quantity_kind_iri': {
                'temperature_C': 'emmo:ThermodynamicTemperature',
                'time_h':        'emmo:Time',
            },
        },
        'results': {
            'hardness_HV': hardness_HV,
            'quantity_kind_iri': {
                # no core-EMMO concept for hardness exists (see note above) —
                # a real project would point this at a domain ontology instead
                'hardness_HV': 'local:VickersHardness',
            },
        },
    }


# Try it on one sample
example = make_sample_record('TS-001', temperature_C=850, time_h=1.0,
                              quench_medium='oil', hardness_HV=612)
print(example['identifier'])
print(example['process'])

## Step 3 — Generate the Dataset

Twelve samples, quenched in either oil or water, at a few different
austenitising temperatures and hold times — synthetic data standing in for
what would otherwise come from a real heat-treatment/hardness-testing
campaign. `quench_medium` and `temperature_C` deliberately drive
`hardness_HV` (water quenching and higher temperature both raise hardness
here) so the pattern is visible if you plot or group the data in Part III.

In [ ]:
import numpy as np

rng = np.random.default_rng(42)

conditions = [
    # (temperature_C, time_h, quench_medium)
    (800, 0.5, 'oil'),   (800, 1.0, 'oil'),   (800, 1.0, 'water'),
    (825, 0.5, 'oil'),   (825, 1.0, 'oil'),   (825, 1.0, 'water'),
    (850, 0.5, 'oil'),   (850, 1.0, 'oil'),   (850, 1.0, 'water'),
    (875, 0.5, 'oil'),   (875, 1.0, 'oil'),   (875, 1.0, 'water'),
]

def simulate_hardness(temperature_C, time_h, quench_medium):
    base = 480 + 0.35 * (temperature_C - 800)
    medium_bonus = 60 if quench_medium == 'water' else 0
    time_bonus = 15 * time_h
    noise = rng.normal(0, 8)
    return round(base + medium_bonus + time_bonus + noise, 1)

dataset = []
for i, (T, t, medium) in enumerate(conditions, start=1):
    hv = simulate_hardness(T, t, medium)
    record = make_sample_record(f'TS-{i:03d}', T, t, medium, hv)
    dataset.append(record)

print(f'Generated {len(dataset)} records')
print(dataset[0])

## Step 4 — Validate Every Record

Reuse the structural validator from Notebook 4 (Section 4.6) — copied below
so this notebook is self-contained — and check every record before trusting
any of them downstream. Validating in a loop like this, rather than only
checking that `len(dataset) == 12`, is what catches the one broken record
among many that a spot check would miss.

In [ ]:
def validate_entry(entry, schema):
    """Structural validator: checks required keys and types against a
    nested schema dict of the form {key: type_or_nested_schema}."""
    errors = []

    def _check(data, schema, path=''):
        for key, expected in schema.items():
            full_path = f'{path}.{key}' if path else key
            if key not in data:
                errors.append(f'MISSING: {full_path}')
                continue
            if isinstance(expected, dict):
                if isinstance(data[key], dict):
                    _check(data[key], expected, full_path)
                else:
                    errors.append(f'EXPECTED nested object at {full_path}')
            elif not isinstance(data[key], expected):
                errors.append(
                    f'WRONG TYPE at {full_path}: expected {expected.__name__}, '
                    f'got {type(data[key]).__name__}'
                )

    _check(entry, schema)
    return errors


all_errors = {}
for record in dataset:
    errors = validate_entry(record, sample_schema)
    if errors:
        all_errors[record['sample_id']] = errors

if all_errors:
    print('Records with validation errors:')
    for sid, errs in all_errors.items():
        print(f'  {sid}: {errs}')
else:
    print(f'All {len(dataset)} records passed validation.')

## Step 5 — Export: Clean CSV + Full JSON Archive

Mirror NOMAD's own split from Notebook 4: a **flat, analysis-ready table**
(what Part III's statistical tests actually need — plain columns, no
nesting) and a **full JSON archive** that keeps every field, including the
EMMO tags and provenance, so nothing is lost. This is the same "don't
throw away the metadata just because pandas wants flat columns" principle
Notebook 1 opened with — you keep both, rather than picking one.

In [ ]:
import json
import pandas as pd

# ── Flat table for analysis (Part III onward) ─────────────────────────────────
rows = [{
    'sample_id':      r['sample_id'],
    'temperature_C':  r['process']['temperature_C'],
    'time_h':         r['process']['time_h'],
    'quench_medium':  r['process']['quench_medium'],
    'hardness_HV':    r['results']['hardness_HV'],
} for r in dataset]

df = pd.DataFrame(rows)
df.to_csv('tool_steel_hardness.csv', index=False)
print(df.head())
print(f'\nSaved {len(df)} rows to tool_steel_hardness.csv')

# ── Full JSON archive, with metadata and EMMO tags preserved ──────────────────
archive = {
    'dataset': {
        'identifier': f'urn:uuid:{uuid.uuid4()}',
        'title':      'Quenched-and-tempered tool-steel hardness dataset',
        'creator':    'A. Lindqvist, Dept. of Chemistry, Uppsala University',
        'created':    date.today().isoformat(),
        'licence':    'CC-BY-4.0',
        'keywords':   ['tool steel', 'quenching', 'hardness', 'heat treatment'],
    },
    'records': dataset,
}
with open('tool_steel_hardness_archive.json', 'w', encoding='utf-8') as f:
    json.dump(archive, f, indent=2)
print(f'Saved full archive with {len(dataset)} records to tool_steel_hardness_archive.json')

## Step 6 — Round-trip Check

Never trust a save without confirming you can load it back correctly — this
is the same discipline Notebook 1 introduced in Section 1.4. Two checks:
the flat CSV should reproduce the DataFrame exactly, and the JSON archive
should still contain the concept tags (EMMO and local) that the CSV, by
design, threw away.

In [ ]:
import json

# CSV round trip
df_reloaded = pd.read_csv('tool_steel_hardness.csv')
print('CSV round trip OK:', df_reloaded.equals(df))

# JSON round trip — and confirm the metadata the CSV dropped is still there
with open('tool_steel_hardness_archive.json', encoding='utf-8') as f:
    archive_reloaded = json.load(f)

print('JSON round trip OK:', len(archive_reloaded['records']) == len(dataset))
first = archive_reloaded['records'][0]
print('Concept tag preserved in archive:',
      first['results']['quantity_kind_iri']['hardness_HV'])
print('(That tag is *not* recoverable from the CSV alone — this is why both files matter.)')

## Step 7 — Dataset-Level FAIR Check

Finally, close the loop with the `fair_check` idea from Notebook 1, applied
this time to the **dataset as a whole** rather than a single measurement.
A dataset is Findable/Accessible/Reusable largely through the fields you
put in `archive['dataset']` above — check that they are all there before
considering the dataset "done."

In [ ]:
def fair_check(record, required_fields=('identifier', 'title', 'licence',
                                       'creator', 'created', 'keywords')):
    return {field: (field in record and record[field] not in (None, '', []))
            for field in required_fields}


report = fair_check(archive['dataset'])
n_ok = sum(report.values())
print(f'Dataset-level FAIR self-check: {n_ok}/{len(report)} fields present')
for field, ok in report.items():
    flag = '✓' if ok else '✗ MISSING'
    print(f'  {flag}  {field}')

## Where This Leads

`tool_steel_hardness.csv` is now exactly the kind of file Part III's
notebooks open with `pd.read_csv` — try loading it there and computing
group means for `oil` vs. `water` quenching, or an ANOVA across
`temperature_C` levels, once you reach Part III, Notebook 4. The archive
file travels with it as the permanent, FAIR, EMMO-tagged record of exactly
how that CSV came to exist — the piece a spreadsheet alone can never
provide.

---
## Exercises

1. **Break it on purpose**: Add one more record to `dataset` by hand (not
   through `make_sample_record`) that is missing `results.hardness_HV`.
   Re-run Step 4's validation loop and confirm it is caught.

2. **New quantity**: Add a `tempering_temperature_C` field to the `process`
   section of `sample_schema` and to `make_sample_record`, with its own
   `quantity_kind_iri` entry (`'emmo:ThermodynamicTemperature'` again — the
   same EMMO concept can label more than one field). Regenerate the dataset
   and re-validate.

3. **Your own dataset**: Pick a small dataset from your own project idea, or
   one of the [course project](../06_project/project.md)'s suggested
   sources. Sketch a `schema` dict and a `make_*_record` function for it,
   following Steps 1–2 above. You do not need to run Steps 3–7 on real data
   yet — the goal is just practising the schema-design step.